In [3]:
import math
import pyomo.environ as pyo
from pyomo.contrib.piecewise import PiecewiseLinearFunction

# ===== 真目标（符号版要用 pyo.sin/pyo.cos） =====
def f_true_symbolic(X, Y):
    return pyo.sin(X) + 0.3*pyo.cos(2*Y) + 0.2*X*Y + 0.2*Y**2

def f_true_numeric(x, y):
    return math.sin(x) + 0.3*math.cos(2*y) + 0.2*x*y + 0.2*y*y

# ===== 网格 & 三角剖分 =====
def make_grid(xmin, xmax, ymin, ymax, nx, ny):
    xs = [xmin + i*(xmax-xmin)/(nx-1) for i in range(nx)]
    ys = [ymin + j*(ymax-ymin)/(ny-1) for j in range(ny)]
    vals = {(x,y): f_true_numeric(x,y) for y in ys for x in xs}
    return xs, ys, vals

def triangulate_rect_grid(xs, ys):
    simplices = []
    nx, ny = len(xs), len(ys)
    for j in range(ny-1):
        for i in range(nx-1):
            p00 = (xs[i],   ys[j])
            p10 = (xs[i+1], ys[j])
            p01 = (xs[i],   ys[j+1])
            p11 = (xs[i+1], ys[j+1])
            simplices.append([p00, p10, p11])
            simplices.append([p00, p11, p01])
    return simplices

# ===== 三点拟合平面 z=ax+by+c，返回 (a,b,c) =====
def fit_plane_through_triangle(tri_pts, val_dict):
    (x1,y1),(x2,y2),(x3,y3) = tri_pts
    z1, z2, z3 = val_dict[(x1,y1)], val_dict[(x2,y2)], val_dict[(x3,y3)]
    det = x1*(y2 - y3) - y1*(x2 - x3) + (x2*y3 - y2*x3)
    assert abs(det) > 1e-14, "退化三角形"
    def det3(a11,a12,a13, a21,a22,a23, a31,a32,a33):
        return a11*(a22*a33 - a23*a32) - a12*(a21*a33 - a23*a31) + a13*(a21*a32 - a22*a31)
    a = det3(z1,y1,1,  z2,y2,1,  z3,y3,1) / det
    b = det3(x1,z1,1,  x2,z2,1,  x3,z3,1) / det
    c = det3(x1,y1,z1, x2,y2,z2, x3,y3,z3) / det
    return (a,b,c)

# ===== 构建 PWLF：显式分片 + 线性函数；同时返回每片 (a,b,c) =====
def build_pwlf(xs, ys, vals):
    simplices = triangulate_rect_grid(xs, ys)
    lin_funcs, coefs = [], []
    for tri in simplices:
        a,b,c = fit_plane_through_triangle(tri, vals)
        lin_funcs.append(lambda X,Y,aa=a,bb=b,cc=c: aa*X + bb*Y + cc)
        coefs.append((a,b,c))
    pw = PiecewiseLinearFunction(
        simplices = simplices,
        linear_functions = lin_funcs,
        triangulation = None
    )
    return pw, simplices, coefs

# ===== 主模型：先把 pw 挂到模型上，再调用 =====
def build_main_model(pw):
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(0,1))
    m.y = pyo.Var(bounds=(0,1))
    m.pw = pw                     # ← 关键：把组件挂到模型
    pw_expr = m.pw(m.x, m.y)      # ← 现在才调用
    m.As = pyo.Expression(expr=pw_expr)
    m.obj = pyo.Objective(expr=f_true_symbolic(m.x, m.y))
    return m

# ===== 每片范围内最小化 [obj - As]（符号表达式写法）=====
def per_piece_min_obj_minus_As(simplices, coefs, solver="ipopt", tee=False):
    out = []
    for tri, (a,b,c) in zip(simplices, coefs):
        mm = pyo.ConcreteModel()
        mm.lmbd = pyo.Var(range(3), bounds=(0,1))
        mm.sum1 = pyo.Constraint(expr = sum(mm.lmbd[i] for i in range(3)) == 1)

        xs = [tri[i][0] for i in range(3)]
        ys = [tri[i][1] for i in range(3)]
        mm.X = pyo.Expression(expr = sum(xs[i]*mm.lmbd[i] for i in range(3)))
        mm.Y = pyo.Expression(expr = sum(ys[i]*mm.lmbd[i] for i in range(3)))

        mm.As = pyo.Expression(expr = a*mm.X + b*mm.Y + c)
        mm.obj = pyo.Objective(expr = f_true_symbolic(mm.X, mm.Y) - mm.As,
                               sense=pyo.minimize)

        opt = pyo.SolverFactory(solver)
        opt.solve(mm, tee=tee)

        x_star = pyo.value(mm.X)
        y_star = pyo.value(mm.Y)
        min_diff = pyo.value(mm.obj)
        out.append((min_diff, x_star, y_star))
    return out

# ===== 串起来跑 =====
if __name__ == "__main__":
    xs, ys, vals = make_grid(0.0, 1.0, 0.0, 1.0, nx=6, ny=5)
    pw, simplices, coefs = build_pwlf(xs, ys, vals)

    m = build_main_model(pw)  # 不需要求解主模型也行
    per_piece_min = per_piece_min_obj_minus_As(simplices, coefs, solver="ipopt", tee=False)

    print("num pieces:", len(simplices))
    print("per-piece min(obj - As):", [v[0] for v in per_piece_min])




ipopt


ApplicationError: No executable found for solver 'ipopt'